# FIP 00 — Fiber photometry exploration

A learning notebook for **accessing and plotting fiber photometry (FIP) data**, as a first
step toward correlating the photometry signals with tongue kinematics.

**Where this runs:** Code Ocean. The data lives in a Code Ocean data asset, not locally.
Attach the asset so it mounts under `/root/capsule/data/`.

- Data asset ID: `6babbf3d-6970-4456-aab5-d730ed57c269`

**Data format:** this asset is a **saved parquet hierarchy** (the output of
`save_nwb_list`), *not* raw `.nwb` files. We load it with Rachel's library
(`rachel_analysis_utils.nwb_utils.load_nwb_list`), which expects:

```
plot_loc/
    df_sess*.csv            (optional)
    rpe_slope.csv           (optional)
    <subject_id>/
        <session_id>/
            df_events.parquet
            df_fip.parquet      # columns: ses_idx, event, data, timestamps, ...
            df_trials.parquet
```

**FIP signal layout:** each row of `df_fip` belongs to one `event` series named
`{channel}_{fiber}` (+ processed variants like `*_dff`):
- **channels**: `G` (green / GCaMP), `R` (red), `Iso` (isosbestic control)
- **fibers**: `0`–`4` (the implanted fiber / brain region)

**Curation toggle:** `USE_CURATION` (section 3) switches between two versions — raw traces
by fiber number, or `apply_curation_nwb_list` with a curation JSON that maps fibers to brain
regions and drops bad channels/sessions.

In [1]:
import os
import glob
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Rachel's lab utilities (installed / mounted in the Code Ocean capsule).
from rachel_analysis_utils import nwb_utils as r_utils

%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 4)

ModuleNotFoundError: No module named 'rachel_analysis_utils'

In [ ]:
# load_nwb_list reads the parquet hierarchy with pd.read_parquet(..., engine="fastparquet"),
# which is hardcoded in rachel_analysis_utils. The capsule image doesn't ship fastparquet,
# so install it on first run. (No-op once it's importable.) For a permanent fix, add
# `fastparquet` to the Code Ocean environment instead.
try:
    import fastparquet  # noqa: F401
except ModuleNotFoundError:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "fastparquet"])
    import fastparquet  # noqa: F401
print("fastparquet", fastparquet.__version__)

## 1. Point at the saved parquet hierarchy

`load_nwb_list(plot_loc)` wants a root directory laid out as
`plot_loc/<subject>/<session>/df_fip.parquet`. The data for this project lives at
`/root/capsule/data/DA_NE_4channels/` (212 sessions), so we hardwire `plot_loc` rather than
globbing the whole asset every session. Flip `RELOCATE=True` to rediscover it by glob if the
asset layout ever changes.

In [ ]:
# Hardwired to the attached asset's saved parquet hierarchy so we skip the
# recursive glob every session. This is the `plot_loc` root that load_nwb_list
# wants: plot_loc/<subject>/<session>/df_fip.parquet
plot_loc = "/root/capsule/data/DA_NE_4channels/"

assert os.path.isdir(plot_loc), (
    f"{plot_loc} not found. Attach asset 6babbf3d-6970-4456-aab5-d730ed57c269 "
    "(saved parquet hierarchy) to this capsule, or fix the path."
)
print("plot_loc =", plot_loc)

# Fallback: if the layout ever changes, set RELOCATE=True to rediscover by glob.
RELOCATE = False
if RELOCATE:
    fip_files = glob.glob(os.path.join("/root/capsule/data", "**", "df_fip.parquet"), recursive=True)
    plot_locs = sorted({os.path.dirname(os.path.dirname(os.path.dirname(f))) for f in fip_files})
    print(f"{len(fip_files)} session(s) found across {len(plot_locs)} root(s):", plot_locs)

## 2. Load sessions with `load_nwb_list`

Returns a list of `dummy_nwb` objects (each with `.df_fip`, `.df_trials`, `.df_events`,
`.session_id`) plus optional session/slope tables. `load_fip=True` is required for the
photometry dataframes.

In [ ]:
nwb_list_raw, df_sess, df_slope = r_utils.load_nwb_list(plot_loc, load_fip=True)

print(f"Loaded {len(nwb_list_raw)} session(s):")
for nwb in nwb_list_raw:
    has_fip = getattr(nwb, "df_fip", None) is not None
    print(f"  {nwb.session_id:<32s} fip={has_fip}")

## 3. Curation — the toggle

Two versions, switched by `USE_CURATION`:

- **`False`** — use the raw `nwb_list`. Traces are labelled by fiber number only. Simplest
  for learning the data layout.
- **`True`** — run `apply_curation_nwb_list` with a curation JSON. This annotates
  `df_fip.intended_measurement` (fiber → brain region), renames `df_trials` columns, drops
  bad channels, and removes flagged sessions.

⚠️ `CeA_datacuration_firstpass.json` is **project-specific** (BWNM central-amygdala). It is
only correct if your fibers match that mapping — otherwise make your own curation file with
the same schema and point `CURATION_FILE` at it.

In [ ]:
USE_CURATION = False
CURATION_FILE = "CeA_datacuration_firstpass"

if USE_CURATION:
    # Locate the curation JSON inside the installed rachel package, with a /src fallback.
    json_path = None
    try:
        import rachel_analysis_utils
        pkg_dir = os.path.dirname(rachel_analysis_utils.__file__)
        cand = os.path.join(pkg_dir, "data_curation", CURATION_FILE + ".json")
        if os.path.exists(cand):
            json_path = cand
    except Exception:
        pass
    if json_path is None:
        hits = glob.glob("/src/**/data_curation/" + CURATION_FILE + ".json", recursive=True)
        assert hits, "Could not locate " + CURATION_FILE + ".json"
        json_path = hits[0]

    with open(json_path, "r") as fh:
        df_curation = json.load(fh)
    print("Using curation:", json_path)

    nwb_list, nwb_list_curated = r_utils.apply_curation_nwb_list(
        nwb_list_raw, df_curation, drop_borderline=True
    )
    print(f"Curated -> {len(nwb_list)} session(s) kept.")
else:
    nwb_list = nwb_list_raw
    print("No curation applied; using raw nwb_list.")

## 4. Pick a session and inspect `df_fip`

Each `dummy_nwb` holds one session's tidy `df_fip`: columns `[ses_idx, event, data,
timestamps, ...]`, where `timestamps` are aligned so **t=0 is the first go cue**. Each unique
`event` is one (channel, fiber, variant) trace.

In [ ]:
SESSION_IDX = 0
nwb = nwb_list[SESSION_IDX]
df_fip = nwb.df_fip

print("session_id:", nwb.session_id)
print("df_fip:", df_fip.shape, "| columns:", list(df_fip.columns))
df_fip.head()

In [ ]:
events = sorted(df_fip["event"].unique())


def parse_event(name):
    """Split a FIP series name into (channel, fiber, variant).

    e.g. 'G_1_dff-poly' -> ('G', '1', 'dff-poly'); 'R_0' -> ('R', '0', 'raw').
    """
    parts = name.split("_")
    channel = parts[0]
    fiber = parts[1] if len(parts) > 1 else "?"
    variant = "_".join(parts[2:]) if len(parts) > 2 else "raw"
    return channel, fiber, variant


meta = pd.DataFrame(
    [(e,) + parse_event(e) for e in events],
    columns=["event", "channel", "fiber", "variant"],
)

# After curation, df_fip carries region labels in `intended_measurement`.
if "intended_measurement" in df_fip.columns:
    ev2region = (
        df_fip.dropna(subset=["intended_measurement"])
        .groupby("event")["intended_measurement"]
        .agg(lambda s: s.mode().iloc[0] if not s.mode().empty else None)
    )
    meta["region"] = meta["event"].map(ev2region)

print(f"{len(events)} FIP series")
meta

## 5. Plot full-session traces

Overlay G/R/Iso for one fiber. Iso is the isosbestic control — motion / bleaching artefacts
show up in both G and Iso, while real calcium transients appear in green only.

In [ ]:
def get_trace(df_fip, event_name):
    """Return (t, y) sorted by time for one FIP series."""
    sub = df_fip[df_fip["event"] == event_name].sort_values("timestamps")
    return sub["timestamps"].to_numpy(), sub["data"].to_numpy()


# Pick the most common fiber and prefer a dff variant if present.
FIBER = meta["fiber"].mode().iloc[0]
variants_here = meta.loc[meta["fiber"] == FIBER, "variant"].unique()
VARIANT = next((v for v in variants_here if "dff" in v), variants_here[0])

region = None
if "region" in meta.columns:
    rvals = meta.loc[meta["fiber"] == FIBER, "region"].dropna().unique()
    region = rvals[0] if len(rvals) else None
label_suffix = (" (" + str(region) + ")") if region else ""
print(f"Plotting fiber {FIBER}{label_suffix}, variant '{VARIANT}'")

fig, ax = plt.subplots()
for ch, color in [("G", "green"), ("R", "red"), ("Iso", "gray")]:
    match = meta[(meta["channel"] == ch) & (meta["fiber"] == FIBER) & (meta["variant"] == VARIANT)]
    if match.empty:
        continue
    t, y = get_trace(df_fip, match["event"].iloc[0])
    ax.plot(t, y, color=color, lw=0.6, alpha=0.8, label=ch)

ax.set_xlabel("time from first go cue (s)")
ax.set_ylabel(f"signal ({VARIANT})")
ax.set_title(f"Fiber {FIBER}{label_suffix} — {nwb.session_id}")
ax.legend()
plt.show()

## 6. Overlay behavioural events

Place the photometry in behavioural context using the same session's `df_trials`. We detect
the go-cue column (it differs across processing versions) and mark cues on a short window.

In [ ]:
df_trials = getattr(nwb, "df_trials", None)
cue_col = None
if df_trials is not None:
    cue_col = next(
        (c for c in ["goCue_start_time_in_session", "goCue_start_time"] if c in df_trials.columns),
        None,
    )
    print("df_trials:", df_trials.shape, "| go-cue column:", cue_col)
else:
    print("No df_trials on this session.")

In [ ]:
match = meta[(meta["channel"] == "G") & (meta["fiber"] == FIBER) & (meta["variant"] == VARIANT)]
assert not match.empty, "No green trace for chosen fiber/variant; adjust FIBER/VARIANT."
t, y = get_trace(df_fip, match["event"].iloc[0])

WINDOW = (0, 60)  # seconds from first go cue
m = (t >= WINDOW[0]) & (t <= WINDOW[1])

fig, ax = plt.subplots()
ax.plot(t[m], y[m], color="green", lw=0.8, label=f"G_{FIBER}{label_suffix}")
if cue_col is not None:
    cues = df_trials[cue_col]
    cues = cues[(cues >= WINDOW[0]) & (cues <= WINDOW[1])]
    for c in cues:
        ax.axvline(c, color="k", lw=0.5, alpha=0.4)
    ax.plot([], [], color="k", alpha=0.4, label="go cue")

ax.set_xlim(*WINDOW)
ax.set_xlabel("time from first go cue (s)")
ax.set_ylabel("signal")
ax.set_title(f"Fiber {FIBER}{label_suffix} green with go cues")
ax.legend()
plt.show()

## 7. Peri-event average (a first real analysis)

Align the chosen signal to each go cue and average across trials. FIP samples don't land on
identical offsets each trial, so we interpolate each trial's snippet onto a common grid, then
take mean ± SEM. `peri_event_matrix` is reusable for any event — including movement onset.

In [ ]:
def peri_event_matrix(t, y, event_times, pre=1.0, post=3.0, dt=0.05):
    """Align trace (t, y) to each event time and resample onto a common grid.

    Parameters
    ----------
    t, y : 1D arrays
        Trace timestamps (s) and values, sorted by time.
    event_times : 1D array
        Event times (s) in the same clock as `t`.
    pre, post : float
        Seconds before / after each event to include.
    dt : float
        Grid resolution (s).

    Returns
    -------
    grid : 1D array of offsets (s), length T
    M    : 2D array (n_events x T) of interpolated values (NaN outside coverage)
    """
    grid = np.arange(-pre, post + dt, dt)
    rows = []
    for et in np.asarray(event_times):
        if np.isnan(et):
            continue
        vals = np.interp(et + grid, t, y, left=np.nan, right=np.nan)
        rows.append(vals)
    return grid, (np.vstack(rows) if rows else np.empty((0, len(grid))))


if cue_col is not None:
    event_times = df_trials[cue_col].to_numpy()
    grid, M = peri_event_matrix(t, y, event_times, pre=1.0, post=3.0, dt=0.05)

    mean = np.nanmean(M, axis=0)
    sem = np.nanstd(M, axis=0) / np.sqrt(np.sum(~np.isnan(M), axis=0))

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(grid, mean, color="green", label=f"G_{FIBER}{label_suffix} (n={M.shape[0]})")
    ax.fill_between(grid, mean - sem, mean + sem, color="green", alpha=0.25)
    ax.axvline(0, color="k", lw=0.8, ls="--", label="go cue")
    ax.set_xlabel("time from go cue (s)")
    ax.set_ylabel("signal")
    ax.set_title(f"Peri-go-cue average — fiber {FIBER}{label_suffix}")
    ax.legend()
    plt.show()
else:
    print("No go-cue column — skipping peri-event average.")

## 8. Next: correlate FIP with tongue kinematics

The bridge between photometry and kinematics is **time**: both `df_fip['timestamps']` and the
kinematics events are aligned to the first go cue. So once you have movement-bout onset times
for this session you can reuse `peri_event_matrix` to align photometry to movement onset
instead of the go cue.

Plan:
1. Load the tongue kinematics for **this session** — match on `ses_idx` (printed below).
2. Extract movement-bout onset times (same go-cue-aligned clock).
3. `peri_event_matrix(t, y, bout_onsets, ...)` → photometry aligned to movement onset.
4. Correlate single-trial photometry features (e.g. peak dF/F in a window) against kinematic
   features (e.g. bout amplitude, duration).

In [ ]:
ses_idx = df_fip["ses_idx"].iloc[0]
print("Session index for matching kinematics:", ses_idx)

# Example once you have `bout_onsets` (go-cue-aligned, seconds):
# grid, M = peri_event_matrix(t, y, bout_onsets, pre=0.5, post=1.5, dt=0.02)
# ... then plot mean +/- SEM as in section 7.